# Amazon Product Reviews Classifier with PySpark

Notebook de referencia para entrenar un clasificador binario de sentimiento con PySpark MLlib.

## Configuraci?n

Ajusta `file_path` a la ubicaci?n del CSV en DBFS o en el sistema local.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lower, regexp_replace, when
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.feature import HashingTF, IDF, StopWordsRemover, Tokenizer

spark = SparkSession.builder.appName("AmazonReviewsSentimentClassifier").getOrCreate()
file_path = "/FileStore/tables/Datafiniti_Amazon_Consumer_Reviews_of_Amazon_Products_May19.csv"

## Carga y etiquetado

In [ ]:
reviews = spark.read.csv(file_path, header=True, inferSchema=True, multiLine=True, escape='"')

data = reviews.select(
    col("`reviews.text`").alias("review_text"),
    col("`reviews.rating`").cast("double").alias("rating"),
).dropna(subset=["review_text", "rating"])

data = data.withColumn(
    "label",
    when(col("rating") >= 4, 1.0).when(col("rating") <= 2, 0.0),
).dropna(subset=["label"])

data = data.withColumn("clean_text", lower(col("review_text")))
data = data.withColumn("clean_text", regexp_replace(col("clean_text"), r"[^a-zA-Z\s]", " "))

data.select("review_text", "rating", "label").show(5, truncate=80)

## Pipeline y entrenamiento

In [ ]:
tokenizer = Tokenizer(inputCol="clean_text", outputCol="words")
remover = StopWordsRemover(inputCol="words", outputCol="filtered_words")
hashing_tf = HashingTF(inputCol="filtered_words", outputCol="raw_features", numFeatures=10000)
idf = IDF(inputCol="raw_features", outputCol="features")
classifier = LogisticRegression(featuresCol="features", labelCol="label", maxIter=20)

pipeline = Pipeline(stages=[tokenizer, remover, hashing_tf, idf, classifier])
train, test = data.randomSplit([0.8, 0.2], seed=42)
model = pipeline.fit(train)

## Evaluaci?n

In [ ]:
predictions = model.transform(test)
evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC",
)
auc = evaluator.evaluate(predictions)
print(f"Test AUC: {auc:.4f}")
predictions.select("review_text", "rating", "label", "prediction", "probability").show(10, truncate=80)